In [58]:
import src.datasets.tabular
import src.datasets.utils
from src.datasets.tabular import get_dataset_config
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import pandas as pd
import numpy as np

from src.models import *

eta = 0.3
poly_size = 2
alpha0 = 5
model_class = 'ridge'
CV_add = True


train_frac = 0.3

dataset = 'wine'
randint = np.random.randint(0, 100, size = 1)[0]
dataset_kwargs = {'root_dir': 'src/datasets', 'random_state': randint, 'is_regression': True, 'train_frac': train_frac}

dataset_config = get_dataset_config(dataset, **dataset_kwargs)

loop_times = int((1 - train_frac) / train_frac)
dset = src.datasets.utils.get_dataset(dataset_config)
X_tr, y_tr, X_te, y_te, X_val, y_val = dset.get_data()

X_te = pd.concat([X_te, X_val])
y_te = pd.concat([y_te, y_val])

if poly_size > 1:
    poly = PolynomialFeatures(poly_size)
    X_tr = poly.fit_transform(X_tr)
    X_te = poly.fit_transform(X_te)

else:
    X_tr['intercept'] = 1
    for i in range(loop_times):
        dset.X_te_list[i]['intercept'] = 1
    X_tr = X_tr.to_numpy()
    y_tr = y_tr.to_numpy()

if model_class == 'ridge':
    model = Ridge(fit_intercept = False, alpha = alpha0)
elif model_class == 'linear':      
    model = LinearRegression(fit_intercept = False)
elif model_class == 'nn':
    model = MLPRegressor(input_dim=len(X_tr[0]), num_classes = 1)
    config = {"lr": 0.1, "batch_size": 128, "train_epochs": 10, "hidden_size":16, "dropout_ratio": 0.1}
    model.update(config)

model.fit(X_tr, y_tr)

if CV_add == True:
    fold_num = 5
    # CV
    kf = KFold(n_splits = fold_num, shuffle = True)
    cv_sample = []
    for train_label, test_label in kf.split(X_tr):
        X_tr1 = X_tr[train_label]
        y_tr1 = y_tr[train_label]

        X_te1 = X_tr[test_label]
        y_te1 = y_tr[test_label]
        if model_class == 'ridge':
            model1 = Ridge(fit_intercept = False, alpha = alpha0)
        else:
            model1 = LinearRegression(fit_intercept = False)
        model1.fit(X_tr1, y_tr1)
        y_te1_pred = model1.predict(X_te1)
        loss1 = (y_te1_pred - y_te1) ** 2
        cv_sample.append(np.mean(np.maximum(loss1 - eta, 0)))
    print('CV result', np.mean(cv_sample))


# OIC DEBIASING
y_tr_pred = model.predict(X_tr)
residual = y_tr - y_tr_pred
loss = (y_tr_pred - y_tr) ** 2
in_sample = np.mean(np.maximum(loss - eta, 0))

print('in-sample result', in_sample)

grad_interact_sum = 0
if model_class != 'nn':
    if model_class == 'ridge':
        cov_inv = np.linalg.pinv(np.cov(X_tr.T) + (alpha0 / len(X_tr) * np.identity(len(X_tr[0]))))
    else:
        cov_inv = np.linalg.pinv(np.cov(X_tr.T))

    for i in range(len(X_tr)):
        if loss[i] > eta:
            grad_interact = 2 * residual[i] * X_tr[i]
            grad_interact_sum += -np.dot(cov_inv, X_tr[i] * residual[i]).dot(grad_interact) / (len(X_tr) ** 2)


else:
    #obtain gradient from nn
    pass

print('debiasing result', in_sample - grad_interact_sum)

#OOS result
out_sample = np.zeros(loop_times)
for i in range(loop_times):
    X_te = dset.X_te_list[i]
    y_te = dset.y_te_list[i]
    if poly_size > 1:
        X_te = poly.fit_transform(X_te)
    y_te_pred = model.predict(X_te)
    loss = (y_te_pred - y_te) ** 2

    out_sample[i] = np.mean(np.maximum(loss - eta, 0))
print('OOS result', np.mean(out_sample))
print('debiasing difference', np.mean(out_sample) - in_sample + grad_interact_sum)

if CV_add == True:
    print('CV difference', np.mean(out_sample) - np.mean(cv_sample))





src/datasets
CV result 0.3397198952438634
in-sample result 0.26420013735146414
debiasing result 0.45438079102875123
OOS result 0.3985417650123768
debiasing difference -0.055839026016374416
CV difference 0.05882186976851339


In [26]:
len(X_tr[0])

91

In [28]:
import src.datasets.tabular
import src.datasets.utils
from src.datasets.tabular import get_dataset_config
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

import pandas as pd
import numpy as np

from src.models import *

eta = 0.3
poly_size = 2
alpha0 = 0.01
model_class = 'nn'


train_frac = 0.3

dataset = 'wine'
randint = np.random.randint(0, 100, size = 1)[0]
dataset_kwargs = {'root_dir': 'src/datasets', 'random_state': randint, 'is_regression': True, 'train_frac': train_frac}

dataset_config = get_dataset_config(dataset, **dataset_kwargs)

loop_times = int((1 - train_frac) / train_frac)
dset = src.datasets.utils.get_dataset(dataset_config)
X_tr, y_tr, X_te, y_te, X_val, y_val = dset.get_data()

#X_tr['intercept'] = 1
X_tr = X_tr.to_numpy()

cov_inv_linear = np.linalg.inv(np.dot(X_tr.T, X_tr)/len(X_tr))
cov_inv_ridge = np.linalg.inv(np.dot(X_tr.T, X_tr)/len(X_tr) + (alpha0 / len(X_tr) * np.identity(len(X_tr[0]))))

src/datasets


In [29]:
cov_inv_linear - cov_inv_ridge

array([[ 6.73633439e-05,  2.13543170e-04, -1.69481251e-04,
         3.57796044e-05, -3.49305361e-04,  1.47202734e-05,
        -3.28727984e-05,  1.43292614e-04,  1.48696075e-05,
         9.05579564e-05,  4.65415430e-06,  1.39645589e-05],
       [ 2.13543170e-04,  1.47564156e-03, -1.09424047e-03,
         2.03686919e-04, -2.36969288e-03, -1.21678742e-05,
        -5.58308229e-05,  9.92962833e-04,  1.29071433e-04,
         5.98089919e-04, -4.60355337e-05,  6.88769304e-04],
       [-1.69481251e-04, -1.09424047e-03,  8.42670851e-04,
        -1.54299268e-04,  1.78301771e-03,  1.17520089e-06,
         5.23436909e-05, -7.40944064e-04, -8.73074256e-05,
        -4.47924208e-04,  2.82313243e-05, -5.02066267e-04],
       [ 3.57796044e-05,  2.03686919e-04, -1.54299268e-04,
         4.12043254e-05, -3.26473699e-04, -2.94150003e-06,
        -1.81159283e-05,  1.36306991e-04,  1.37688716e-05,
         8.03063998e-05, -3.58861039e-06,  5.78616619e-05],
       [-3.49305361e-04, -2.36969288e-03,  1.7830177

In [7]:
import torch
from torch.autograd.functional import hessian
from torch.func import functional_call

model = torch.nn.Linear(2, 2)
inp = torch.rand(1, 2)
criterion = torch.nn.CrossEntropyLoss()

def loss(*params):
    out: torch.Tensor = functional_call(model, {n: p for n, p in zip(names, params)}, inp)
    return criterion (out, torch.zeros(len(inp), dtype=torch.long))

names = list(n for n, _ in model.named_parameters())
result = hessian(loss, tuple(model.parameters()))
print(torch.inverse(result))


TypeError: inverse(): argument 'input' (position 1) must be Tensor, not tuple

In [25]:
import torch
import torch.nn as nn

a = torch.ones(3, 4)
module = nn.Linear(4, 16)

def func(weight, bias):
    del module.weight
    module.weight = weight
    del module.bias
    module.bias = bias
    return module(a)

result = torch.autograd.functional.jacobian(func, tuple(module.parameters()))
print(result[0].shape)  # weight (3, 16, 16, 4)
print(result[1].shape)  # bias   (3, 16, 16)
print(result[0][0].shape)  # Correct Values!

torch.Size([3, 16, 16, 4])
torch.Size([3, 16, 16])
torch.Size([16, 16, 4])


In [14]:
import torch
from torch.autograd.functional import jacobian, hessian
from torch.func import functional_call

model = torch.nn.Linear(2, 2)
inp = torch.rand(1, 2)

#names = list(n for n, _ in model.named_parameters())

def loss(*params):
    model2 = torch.nn.Linear(2, 2)
    out: torch.Tensor = functional_call(model, {n: p for n, p in zip(names, params)}, inp)
    return out.square().sum()


names = list(n for n, _ in model.named_parameters())
result = hessian(loss, tuple(model.parameters()))
print(result[0][0].reshape(4, 4))
print(result[1][1].shape)

TypeError: linear(): argument 'input' (position 1) must be Tensor, not Linear

In [1]:
import torch
from torch.autograd.functional import hessian, jacobian
from torch.func import functional_call
from src.models import *

model = MLP2(4, 1)
inp = torch.rand(20, 4)

# return torch.one array jacob
def jacob_format(result_jacob):
    transform_jacobs = []
    for i in range(len(result_jacob)):
        transform_jacob = result_jacob[i].reshape(1, -1)
        if len(transform_jacob.size()) > 1:
            transform_jacobs.append(transform_jacob[0])
        else:
            transform_jacobs.append(transform_jacob)
    node_nums = [len(transform_jacobs[i]) for i in range(len(result_jacob))]
    output_jacob = torch.concat([transform_jacobs[i] for i in range(len(result_jacob))])
    return node_nums, output_jacob

def hessian_format(result_hessian):
    a = []
    for j, node_num in enumerate(node_nums):
        a.append(torch.concat([result_hessian[j][i].reshape(node_num, -1) for i, __ in enumerate(node_nums)], dim = 1))
    final_hessian = torch.concat([a[i] for i in range(len(node_nums))])
    return final_hessian
criterion = torch.nn.MSELoss()

gradients = []
hessians = []
for i in range(10):
    def loss_i(*params):
        out: torch.Tensor = functional_call(model, {n: p for n, p in zip(names, params)}, inp[i])
        return torch.square(out - torch.zeros(1)).sum()

    names = list(n for n, _ in model.named_parameters())
    result_jacob = jacobian(loss_i, tuple(model.parameters()))
    node_nums, a = jacob_format(result_jacob)
    result_hessian = hessian(loss_i, tuple(model.parameters()))
    b = hessian_format(result_hessian)
    gradients.append(a)
    hessians.append(b)
    

# # overall hessian evaluation
# def loss_i(*params):
#     out: torch.Tensor = functional_call(model, {n: p for n, p in zip(names, params)}, inp)
#     return criterion (out, torch.zeros(len(inp), dtype=torch.float))

# names = list(n for n, _ in model.named_parameters())


# hessian


#result_hessian = hessian(loss, tuple(model.parameters()))
#print(result[5])
# print(result[0][0].reshape(64, 64))
# print(result[1][1])

In [3]:
node_nums

[64, 16, 256, 16, 16, 1]

In [57]:
for i, node_num in enumerate(node_nums):
    print(result_hessian[0][i].reshape(-1, node_num).shape)

torch.Size([64, 64])
torch.Size([64, 16])
torch.Size([64, 256])
torch.Size([64, 16])
torch.Size([64, 16])
torch.Size([64, 1])


In [3]:
import torch
a = torch.zeros(1)
a + 3

tensor([3.])

In [23]:
c = torch.zeros(1)
a = torch.Tensor([[1,3],[1, 3]])
b = torch.Tensor([2,4])
c += torch.mm(b.reshape(1, -1), a).mm(b.reshape(-1, 1))[0]

In [24]:
torch.inverse(a)

_LinAlgError: linalg.inv: The diagonal element 2 is zero, the inversion could not be completed because the input matrix is singular.

In [16]:
torch.mm(b.reshape(1, -1), a).mm(b.reshape(-1, 1))

tensor([[140.]])

In [68]:
a = []
for j, node_num in enumerate(node_nums):
    a.append(torch.concat([result_hessian[j][i].reshape(node_num, -1) for i, __ in enumerate(node_nums)], dim = 1))
final_hessian = torch.concat([a[i] for i in range(len(node_nums))])



In [67]:
node_nums = [64, 16, 256, 16, 16, 1]
for i, __ in enumerate(node_nums):
    print(result_hessian[j][i].reshape(node_num, -1).shape)


torch.Size([64, 64])
torch.Size([64, 16])
torch.Size([64, 256])
torch.Size([64, 16])
torch.Size([64, 16])
torch.Size([64, 1])


In [70]:
final_hessian[15:20, 15:20]

tensor([[ 3.4510e-05, -1.6916e-05, -1.0090e-05, -1.7027e-05, -7.9513e-06],
        [-1.6916e-05, -8.6062e-05, -5.1331e-05, -8.6623e-05, -4.0452e-05],
        [-1.0090e-05, -5.1331e-05, -3.0616e-05, -5.1666e-05, -2.4128e-05],
        [-1.7027e-05, -8.6623e-05, -5.1666e-05, -8.7188e-05, -4.0716e-05],
        [-7.9513e-06, -4.0452e-05, -2.4128e-05, -4.0716e-05, -1.9014e-05]])